In [1]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [2]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git



Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 99 (delta 43), reused 62 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 500.90 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [3]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [4]:
!git config --global credential.helper store

In [5]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")


GitHub authentication configured.


In [6]:
!git fetch origin
!git switch nehna

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [7]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

nothing to commit, working tree clean


In [8]:
!pip install -q datasets

In [12]:
import requests
import json

url = "https://huggingface.co/datasets/ai4bharat/IndicQA/resolve/main/data/indicqa.ml.json?download=true"

response = requests.get(url)
response.raise_for_status()

indicqa_raw = response.json()

print("Downloaded successfully!")
print(indicqa_raw.keys())

Downloaded successfully!
dict_keys(['version', 'data'])


In [13]:
print("Number of articles:", len(indicqa_raw["data"]))

print("\nFirst article keys:")
print(indicqa_raw["data"][0].keys())

Number of articles: 247

First article keys:
dict_keys(['title', 'paragraphs'])


In [14]:
article = indicqa_raw["data"][0]

print("\nNumber of paragraphs in first article:")
print(len(article["paragraphs"]))

print("\nFirst paragraph:")
print(article["paragraphs"][0])


Number of paragraphs in first article:
1

First paragraph:
{'context': ' . ഹരപ്പയുമായി യാതൊരു ബന്ധവും കാണുന്ന തരത്തിലല്ല അവയുടെ രീതി. ഹരപ്പൻ കോട്ടക്ക്\u200c തെക്ക്\u200cഭാഗത്തായി കണ്ടെടുത്ത  ശ്മശാനസംസ്കൃതിയുടെ ഭാഗങ്ങളാകട്ടെ ഹരപ്പൻ സംസ്കാരവുമായി പൊരുത്തപ്പെട്ടു പോവാത്തതും പ്രകടമായ വ്യത്യാസമുള്ളവയുമാണ്\u200c. . ഇതിനെ ഹരപ്പാനന്തരഘട്ടമായി ചരിത്രകാരന്മാർ കണക്കാക്കുന്നു. ഇവിടെ നിന്നും കിട്ടിയ മൺപാത്രങ്ങളും അവയിലെ ചിത്രലേഖനങ്ങളും ഹരപ്പൻ പ്രദേശത്തിനു പുറത്തു നിന്നുള്ള ഒരു ജനവിഭാഗത്തിന്റേത്\u200c എന്ന് സംശയിക്കത്തക്കവിധം വ്യത്യസ്തങ്ങളായിരുന്നു. ഹരപ്പൻ അധിവാസകേന്ദ്രങ്ങളിലേക്ക്\u200c ഏതോ പരദേശിജീവിതരീതിയുടെ കടന്നു കയറ്റത്തേയാണ് ഇത്\u200c സൂചിപ്പിക്കുന്നത്\u200c. അടുത്തകാലത്തായി നടത്തിയ ഗവേഷണങ്ങളിൽ ശവശരീരങ്ങൾ കണ്ടെത്തിയത് കൂട്ടക്കൊലകാരണമല്ല മറിച്ച് രോഗങ്ങൾ മൂലം മരണപ്പെട്ടവരുടേതാണെന്നാണ്\u200c തെളിഞ്ഞത്. അസ്ഥികൂടങ്ങളുടെ പഠനത്തിൽ നിന്ന് മരണം ഉപരോധാമോ മറ്റോ കാരണമായി പിടിപെട്ട അനീമിയ പോലുള്ള അസുഖങ്ങൾ മൂലമാണ്\u200c ഉണ്ടായതെന്ന് കണ്ടെത്തിട്ടുണ്ട്. മറ്റൊരു കൂട്ടം ഗവേഷകരുടേ അഭിപ്രായത്തിൽ  ഹരപ്പൻ നാഗരികതയ

In [15]:
malayalam_passages = []
qa_rows = []

for article in indicqa_raw["data"]:
    for paragraph in article["paragraphs"]:

        context = paragraph["context"].strip()

        malayalam_passages.append(context)

        for qa in paragraph["qas"]:
            qa_rows.append({
                "question_id": qa["id"],
                "question": qa["question"].strip(),
                "gold_answers": [
                    answer["text"].strip()
                    for answer in qa["answers"]
                ],
                "context": context
            })

print("Total passages:", len(malayalam_passages))
print("Total QA pairs:", len(qa_rows))

Total passages: 247
Total QA pairs: 1589


In [16]:
print("Example passage:")
print(malayalam_passages[0])

print("\nExample question:")
print(qa_rows[0]["question"])

print("\nGold answer:")
print(qa_rows[0]["gold_answers"])

Example passage:
. ഹരപ്പയുമായി യാതൊരു ബന്ധവും കാണുന്ന തരത്തിലല്ല അവയുടെ രീതി. ഹരപ്പൻ കോട്ടക്ക്‌ തെക്ക്‌ഭാഗത്തായി കണ്ടെടുത്ത  ശ്മശാനസംസ്കൃതിയുടെ ഭാഗങ്ങളാകട്ടെ ഹരപ്പൻ സംസ്കാരവുമായി പൊരുത്തപ്പെട്ടു പോവാത്തതും പ്രകടമായ വ്യത്യാസമുള്ളവയുമാണ്‌. . ഇതിനെ ഹരപ്പാനന്തരഘട്ടമായി ചരിത്രകാരന്മാർ കണക്കാക്കുന്നു. ഇവിടെ നിന്നും കിട്ടിയ മൺപാത്രങ്ങളും അവയിലെ ചിത്രലേഖനങ്ങളും ഹരപ്പൻ പ്രദേശത്തിനു പുറത്തു നിന്നുള്ള ഒരു ജനവിഭാഗത്തിന്റേത്‌ എന്ന് സംശയിക്കത്തക്കവിധം വ്യത്യസ്തങ്ങളായിരുന്നു. ഹരപ്പൻ അധിവാസകേന്ദ്രങ്ങളിലേക്ക്‌ ഏതോ പരദേശിജീവിതരീതിയുടെ കടന്നു കയറ്റത്തേയാണ് ഇത്‌ സൂചിപ്പിക്കുന്നത്‌. അടുത്തകാലത്തായി നടത്തിയ ഗവേഷണങ്ങളിൽ ശവശരീരങ്ങൾ കണ്ടെത്തിയത് കൂട്ടക്കൊലകാരണമല്ല മറിച്ച് രോഗങ്ങൾ മൂലം മരണപ്പെട്ടവരുടേതാണെന്നാണ്‌ തെളിഞ്ഞത്. അസ്ഥികൂടങ്ങളുടെ പഠനത്തിൽ നിന്ന് മരണം ഉപരോധാമോ മറ്റോ കാരണമായി പിടിപെട്ട അനീമിയ പോലുള്ള അസുഖങ്ങൾ മൂലമാണ്‌ ഉണ്ടായതെന്ന് കണ്ടെത്തിട്ടുണ്ട്. മറ്റൊരു കൂട്ടം ഗവേഷകരുടേ അഭിപ്രായത്തിൽ  ഹരപ്പൻ നാഗരികതയുടെ അന്ത്യം കാലാവസ്ഥാ വ്യതിയാനം മൂലമാണ്. തുടർച്ചയായ പ്രളയമോ, അതെത്തുടർന്നുണ്ടായ വനനശീകരണമോ ആയിരിക്കാം 

In [17]:
# Extract all Malayalam passages and QA pairs

malayalam_passages = []
qa_rows = []

for article in indicqa_raw["data"]:
    for paragraph in article["paragraphs"]:

        context = paragraph["context"].strip()

        malayalam_passages.append(context)

        for qa in paragraph["qas"]:
            qa_rows.append({
                "question_id": qa["id"],
                "question": qa["question"].strip(),
                "gold_answers": [
                    answer["text"].strip()
                    for answer in qa["answers"]
                ],
                "context": context
            })

print("Total passages:", len(malayalam_passages))
print("Total QA pairs:", len(qa_rows))

Total passages: 247
Total QA pairs: 1589


In [18]:
print("Example passage:")
print(malayalam_passages[0])

print("\nExample question:")
print(qa_rows[0]["question"])

print("\nGold answer:")
print(qa_rows[0]["gold_answers"])

Example passage:
. ഹരപ്പയുമായി യാതൊരു ബന്ധവും കാണുന്ന തരത്തിലല്ല അവയുടെ രീതി. ഹരപ്പൻ കോട്ടക്ക്‌ തെക്ക്‌ഭാഗത്തായി കണ്ടെടുത്ത  ശ്മശാനസംസ്കൃതിയുടെ ഭാഗങ്ങളാകട്ടെ ഹരപ്പൻ സംസ്കാരവുമായി പൊരുത്തപ്പെട്ടു പോവാത്തതും പ്രകടമായ വ്യത്യാസമുള്ളവയുമാണ്‌. . ഇതിനെ ഹരപ്പാനന്തരഘട്ടമായി ചരിത്രകാരന്മാർ കണക്കാക്കുന്നു. ഇവിടെ നിന്നും കിട്ടിയ മൺപാത്രങ്ങളും അവയിലെ ചിത്രലേഖനങ്ങളും ഹരപ്പൻ പ്രദേശത്തിനു പുറത്തു നിന്നുള്ള ഒരു ജനവിഭാഗത്തിന്റേത്‌ എന്ന് സംശയിക്കത്തക്കവിധം വ്യത്യസ്തങ്ങളായിരുന്നു. ഹരപ്പൻ അധിവാസകേന്ദ്രങ്ങളിലേക്ക്‌ ഏതോ പരദേശിജീവിതരീതിയുടെ കടന്നു കയറ്റത്തേയാണ് ഇത്‌ സൂചിപ്പിക്കുന്നത്‌. അടുത്തകാലത്തായി നടത്തിയ ഗവേഷണങ്ങളിൽ ശവശരീരങ്ങൾ കണ്ടെത്തിയത് കൂട്ടക്കൊലകാരണമല്ല മറിച്ച് രോഗങ്ങൾ മൂലം മരണപ്പെട്ടവരുടേതാണെന്നാണ്‌ തെളിഞ്ഞത്. അസ്ഥികൂടങ്ങളുടെ പഠനത്തിൽ നിന്ന് മരണം ഉപരോധാമോ മറ്റോ കാരണമായി പിടിപെട്ട അനീമിയ പോലുള്ള അസുഖങ്ങൾ മൂലമാണ്‌ ഉണ്ടായതെന്ന് കണ്ടെത്തിട്ടുണ്ട്. മറ്റൊരു കൂട്ടം ഗവേഷകരുടേ അഭിപ്രായത്തിൽ  ഹരപ്പൻ നാഗരികതയുടെ അന്ത്യം കാലാവസ്ഥാ വ്യതിയാനം മൂലമാണ്. തുടർച്ചയായ പ്രളയമോ, അതെത്തുടർന്നുണ്ടായ വനനശീകരണമോ ആയിരിക്കാം 

In [19]:
# Keep unique Malayalam passages while preserving order

unique_passages = list(dict.fromkeys(malayalam_passages))

print("Total passages:", len(malayalam_passages))
print("Unique passages:", len(unique_passages))

# Use up to 1000 passages
selected_passages = unique_passages[:1000]

corpus_ml = [
    {
        "passage_id": i,
        "text": text
    }
    for i, text in enumerate(selected_passages)
]

context_to_id_ml = {
    text: i
    for i, text in enumerate(selected_passages)
}

print("Corpus size:", len(corpus_ml))
print("\nFirst corpus item:")
print(corpus_ml[0])

Total passages: 247
Unique passages: 247
Corpus size: 247

First corpus item:
{'passage_id': 0, 'text': '. ഹരപ്പയുമായി യാതൊരു ബന്ധവും കാണുന്ന തരത്തിലല്ല അവയുടെ രീതി. ഹരപ്പൻ കോട്ടക്ക്\u200c തെക്ക്\u200cഭാഗത്തായി കണ്ടെടുത്ത  ശ്മശാനസംസ്കൃതിയുടെ ഭാഗങ്ങളാകട്ടെ ഹരപ്പൻ സംസ്കാരവുമായി പൊരുത്തപ്പെട്ടു പോവാത്തതും പ്രകടമായ വ്യത്യാസമുള്ളവയുമാണ്\u200c. . ഇതിനെ ഹരപ്പാനന്തരഘട്ടമായി ചരിത്രകാരന്മാർ കണക്കാക്കുന്നു. ഇവിടെ നിന്നും കിട്ടിയ മൺപാത്രങ്ങളും അവയിലെ ചിത്രലേഖനങ്ങളും ഹരപ്പൻ പ്രദേശത്തിനു പുറത്തു നിന്നുള്ള ഒരു ജനവിഭാഗത്തിന്റേത്\u200c എന്ന് സംശയിക്കത്തക്കവിധം വ്യത്യസ്തങ്ങളായിരുന്നു. ഹരപ്പൻ അധിവാസകേന്ദ്രങ്ങളിലേക്ക്\u200c ഏതോ പരദേശിജീവിതരീതിയുടെ കടന്നു കയറ്റത്തേയാണ് ഇത്\u200c സൂചിപ്പിക്കുന്നത്\u200c. അടുത്തകാലത്തായി നടത്തിയ ഗവേഷണങ്ങളിൽ ശവശരീരങ്ങൾ കണ്ടെത്തിയത് കൂട്ടക്കൊലകാരണമല്ല മറിച്ച് രോഗങ്ങൾ മൂലം മരണപ്പെട്ടവരുടേതാണെന്നാണ്\u200c തെളിഞ്ഞത്. അസ്ഥികൂടങ്ങളുടെ പഠനത്തിൽ നിന്ന് മരണം ഉപരോധാമോ മറ്റോ കാരണമായി പിടിപെട്ട അനീമിയ പോലുള്ള അസുഖങ്ങൾ മൂലമാണ്\u200c ഉണ്ടായതെന്ന് കണ്ടെത്തിട്ടുണ്ട്. മറ്റൊരു കൂട്ടം ഗവേഷകരുടേ

In [20]:
import random

random.seed(42)

candidates_ml = []

for row in qa_rows:
    if row["context"] in context_to_id_ml:
        candidates_ml.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "gold_answers": row["gold_answers"],
            "gold_passage_id": context_to_id_ml[row["context"]]
        })

print("Usable candidate questions:", len(candidates_ml))

Usable candidate questions: 1589


In [21]:
questions_ml = random.sample(
    candidates_ml,
    min(100, len(candidates_ml))
)

print("Final Malayalam questions:", len(questions_ml))

Final Malayalam questions: 100


In [22]:
for q in questions_ml:
    passage = corpus_ml[q["gold_passage_id"]]["text"]

    assert any(
        answer in passage
        for answer in q["gold_answers"]
    ), q["question_id"]

print("All answer-in-passage checks passed!")

AssertionError: 864

In [23]:
q = next(q for q in questions_ml if q["question_id"] == "864" or q["question_id"] == 864)

print("QUESTION:")
print(q["question"])

print("\nGOLD ANSWERS:")
print(q["gold_answers"])

print("\nGOLD PASSAGE ID:")
print(q["gold_passage_id"])

print("\nPASSAGE:")
print(corpus_ml[q["gold_passage_id"]]["text"])

QUESTION:
ചോള കാലഘട്ടത്തിലെ വാസ്തുവിദ്യയുടെ ഉത്തമ ഉദാഹരണങ്ങൾ ആയ ക്ഷേത്രങ്ങൾ  ഏതെല്ലാം ?

GOLD ANSWERS:
['.തഞ്ചാവൂരിലേയും, ഗംഗൈകൊണ്ടചോളപുരത്തേയും']

GOLD PASSAGE ID:
136

PASSAGE:
തമിഴ് സാഹിത്യം, വാസ്തുവിദ്യ എന്നിവയുടെ പ്രോത്സാഹകർ ആയിരുന്നു ചോളരാജാക്കന്മാർ. ചോളരുടെ കീഴിൽ കല, മതം, സാഹിത്യം എന്നിവയിൽ തമിഴ് രാജ്യം പുതിയ ഉയരങ്ങളിലെത്തി. ഈ മേഖലകളിലെല്ലാം, പല്ലവരുടെ കീഴിൽ ആരംഭിച്ച പ്രസ്ഥാനങ്ങൾ അവയുടെ പരമോന്നതിയിലെത്തി. വലിയ ക്ഷേത്രങ്ങൾ, ശിലാശില്പങ്ങൾ, വെങ്കലശില്പങ്ങൾ എന്നീ രൂപങ്ങളിലെ വാസ്തുവിദ്യ  ചോളരുടെ കീഴിൽ ഇന്ത്യയിൽ അതുവരെക്കാണാത്ത ഉന്നതിയിലെത്തി. ഇവരുടെ പ്രോത്സാഹനത്തിൽ ആണ് തമിഴ് സാഹിത്യത്തിലെ പല പ്രധാന കൃതികളും തമിഴ്നാട്ടിലെ പല പ്രധാന ക്ഷേത്രങ്ങളും രൂപംകൊണ്ടത്. ക്ഷേത്രനിർമ്മാണത്തെ വളരെ പ്രോത്സാഹിപ്പിച്ച ചോളരാജാക്കന്മാർ ക്ഷേത്രങ്ങളെ ആരാധനാലയങ്ങൾ എന്നതിനു പുറമേ വാണിജ്യകേന്ദ്രങ്ങളായും കരുതി. ജനങ്ങളുടെ ആവാസമേഖലയുടെ കേന്ദ്രങ്ങളായിരുന്നു ക്ഷേത്രങ്ങൾ. ജനവാസകേന്ദ്രങ്ങൾ അവക്കു ചുറ്റുമായാണ് രൂപം കൊണ്ടത്. തഞ്ചാവൂരിലേയും, ഗംഗൈകൊണ്ടചോളപുരത്തേയും ക്ഷേത്രങ്ങൾ ചോളകാലത്തെ വാസ്തുശില്പകലയ്ക്ക് ഉത്തമോദാഹരണങ

In [24]:
import re

def clean_answer(answer):
    answer = answer.strip()

    # Remove accidental leading/trailing punctuation
    answer = re.sub(r'^[\s\.,;:!?]+', '', answer)
    answer = re.sub(r'[\s\.,;:!?]+$', '', answer)

    return answer.strip()

In [25]:
qa_rows = []

for article in indicqa_raw["data"]:
    for paragraph in article["paragraphs"]:

        context = paragraph["context"].strip()

        for qa in paragraph["qas"]:
            qa_rows.append({
                "question_id": qa["id"],
                "question": qa["question"].strip(),
                "gold_answers": [
                    clean_answer(answer["text"])
                    for answer in qa["answers"]
                ],
                "context": context
            })

print("Total QA pairs:", len(qa_rows))

Total QA pairs: 1589


In [26]:
import random

random.seed(42)

candidates_ml = []

for row in qa_rows:
    if row["context"] in context_to_id_ml:
        candidates_ml.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "gold_answers": row["gold_answers"],
            "gold_passage_id": context_to_id_ml[row["context"]]
        })

print("Usable candidate questions:", len(candidates_ml))

questions_ml = random.sample(
    candidates_ml,
    min(100, len(candidates_ml))
)

print("Final Malayalam questions:", len(questions_ml))

Usable candidate questions: 1589
Final Malayalam questions: 100


In [27]:
for q in questions_ml:
    passage = corpus_ml[q["gold_passage_id"]]["text"]

    assert any(
        answer in passage
        for answer in q["gold_answers"]
    ), q["question_id"]

print("All answer-in-passage checks passed!")

All answer-in-passage checks passed!


In [28]:
for i, q in enumerate(questions_ml[:5], 1):
    print("=" * 80)
    print(f"Example {i}")
    print("Question:", q["question"])
    print("Gold answer:", q["gold_answers"])
    print("Gold passage ID:", q["gold_passage_id"])
    print("\nPassage:")
    print(corpus_ml[q["gold_passage_id"]]["text"])
    print()

Example 1
Question: ബുദ്ധമതത്തിന്റെ പുതിയ ശാഖ ഏത്?
Gold answer: ['മഹായാനം']
Gold passage ID: 204

Passage:
യവന ചരിത്രത്തിൽ പെരിക്ലിസിന്റേയും ഇംഗ്ലണ്ടിന്റെ ചരിത്രത്തിൽ എലിസബത്ത് രാജ്ഞി യുടേയും റോമാ ചരിത്രത്തിൽ അഗസ്റ്റസിന്റേയും കാലത്തിന് സമമായാണ് സാംസ്കാരിരംഗത്തെ ചരിത്രകാരന്മാർ വിലയിരുത്തുന്നത്. പ്രശസ്തമായ അജന്താ ഗുഹാക്ഷേത്രത്തിലെ 28 ഗുഹകളിൽ മിക്കവയും ഈ കാലഘട്ടത്തിന്റെ സൃഷ്ടികളാണ്. ബ്രാഹ്മണ മതം ആധുനിക ഹൈന്ദവ മതമായി രൂപാന്തരപ്പെട്ടതാണ് ഇക്കാലത്തെ ഒരു സവിശേഷത. വിഷ്ണു ഭക്തന്മാരായ ഗുപ്തന്മാർ അന്നു വരെ പല വിഷമഘട്ടങ്ങളേയും മറ്റു മതങ്ങളുടെ മാത്സര്യത്തേയും നേരിടേണ്ടിവന്ന ഹിന്ദുമതത്തെ പരിപോഷിപ്പിച്ചു. ഹിന്ദു മതത്തിന്റെ നവീകരണത്തിനും ഇക്കാലം സാക്ഷ്യം വഹിച്ചു. ഹിന്ദു ദൈവങ്ങൾക്ക് വിപ്ലവകരമായ മാറ്റങ്ങൾ വന്നു. പുതിയ ക്ഷേത്രങ്ങളും സ്തംഭങ്ങളും പണികഴിപ്പിക്കപ്പെട്ടു. ഇതിനാൽ ജനങ്ങൾ കൂടുതൽ ഉത്സാഹഭരിതരും ആരാധനയിൽ ശ്രദ്ധയുള്ളവരും ആയി. ബുദ്ധമതത്തിലെ പുതിയ ശാഖയായ മഹായാനം ഇക്കാലത്ത് കൂടുതൽ ഹിന്ദുത്വവത്കരിക്കപ്പെട്ടതായി. ബ്രാഹ്മണന്മാർ ബുദ്ധമതത്തെ ഹിന്ദു മതത്തിന്റെ ശാഖയായി വരെ പ്രഖ്യാപിച്ചും ബുദ്ധമതവും ഹിന്ദു മതവ

In [29]:
candidates_ml = []

for row in qa_rows:
    # Remove empty answers
    valid_answers = [
        answer for answer in row["gold_answers"]
        if answer.strip()
    ]

    if not valid_answers:
        continue

    if row["context"] in context_to_id_ml:
        candidates_ml.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "gold_answers": valid_answers,
            "gold_passage_id": context_to_id_ml[row["context"]]
        })

print("Usable candidate questions:", len(candidates_ml))

Usable candidate questions: 1101


In [30]:
random.seed(42)

questions_ml = random.sample(
    candidates_ml,
    min(100, len(candidates_ml))
)

print("Final Malayalam questions:", len(questions_ml))

Final Malayalam questions: 100


In [34]:
for q in questions_ml:
    passage = normalize_for_match(
        corpus_ml[q["gold_passage_id"]]["text"]
    )

    assert len(q["gold_answers"]) > 0, q["question_id"]

    assert any(
        normalize_for_match(answer) in passage
        for answer in q["gold_answers"]
        if answer.strip()
    ), q["question_id"]

print("All answer-in-passage checks passed!")

All answer-in-passage checks passed!


In [32]:
q = next(q for q in questions_ml if str(q["question_id"]) == "467")

print("QUESTION:")
print(q["question"])

print("\nGOLD ANSWERS:")
print(q["gold_answers"])

print("\nPASSAGE ID:")
print(q["gold_passage_id"])

print("\nPASSAGE:")
print(corpus_ml[q["gold_passage_id"]]["text"])

QUESTION:
ഹാരപ്പൻ ഭാഷ ദ്രാവിഡമാണെന്ന നിഗമനത്തിലേക്ക് നയിച്ചുത്  ഏത് ഗവേഷകന്റെ കണ്ടെത്തലുകൾ ആണ് ?

GOLD ANSWERS:
['ഡോ.സ്വിലെബിൽ']

PASSAGE ID:
71

PASSAGE:
ഈ കണ്ടുപിടിത്തങ്ങൾ നിഗൂഢഭാഷാപാരായണചരിത്രത്തിലെ മൂലക്കല്ലാണ്‌ എന്നാണ്‌ ഡോ. സ്വെലേബിൽ വിശേഷിപ്പിച്ചത്‌. (ദ്രവീഡീയൻ ലിങ്ങ്വിസ്റ്റിക്സ്‌)ഇവരെ കൂടാതെ ഇന്ത്യക്കാരായ ഐരാവതം മഹാദേവൻ, എസ്‌. ആർ. റാവു, അമേരിക്കക്കാരനായ വാൾട്ടർ ഫെർസെർവീസ്‌, കിന്നിയർ-വിൽസൺ എന്നിവരും ഈ ദിശയിൽ പഠനം നടത്തുകയുണ്ടായി. പക്ഷെ ഈ പഠനങ്ങളുടെ ഫലമായി അഭിപ്രായൈക്യത്തേക്കാളുപരി അഭിപ്രായവ്യത്യാസങ്ങളാണുണ്ടായത്. ഡോ. എസ്‌. ആർ. റാവു ലിപി വായിക്കുന്നതിൽ വിജയിച്ചു എന്നും അത്‌ പ്രാഗ്‌സംസ്കൃതമായിരുന്നു എന്നുമാണ്‌ ചിലർ അദ്ദേഹത്തെ ഉദ്ധരിച്ച്‌ പറയുന്നത്‌.  . അദ്ദേഹത്തിനും അങ്ങനെ ഒരഭിപ്രായം ഉണ്ടെങ്കിലും അദ്ദേഹത്തിന്റേത് ഒരു വ്യാഖ്യാനശ്രമം  മാത്രമായിരുന്നു. ഐരാവതം മഹാദേവനെപ്പോലുള്ള പ്രഗല്ഭരായ ഭാഷാവിദഗ്ദ്ധർ റാവുവിന്റെ ശ്രമത്തെ അബദ്ധജടിലമെന്നാണ്‌ വിലയിരുത്തിയത്‌. യു. എസ്‌. എസ്‌. ആർ. അക്കാദമി ഓഫ്‌ സയൻസിന്റെ ആഭിമുഖ്യത്തിൽ രൂപീകതമായ മറ്റൊരു സമിതിയും പഠനങ്ങൾ നടത്തിയവരിൽപ്പെടുന്നു. വടക്കേ അമേരിക്ക

In [33]:
import re

def normalize_for_match(text):
    text = text.strip()

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces around punctuation
    text = re.sub(r'\s*([.,;:!?])\s*', r'\1', text)

    return text

In [35]:
import json

with open("data/corpus_ml.json", "w", encoding="utf-8") as f:
    json.dump(corpus_ml, f, ensure_ascii=False, indent=2)

with open("data/questions_ml.json", "w", encoding="utf-8") as f:
    json.dump(questions_ml, f, ensure_ascii=False, indent=2)

print("Malayalam dataset saved successfully!")
print("Corpus passages:", len(corpus_ml))
print("Questions:", len(questions_ml))

Malayalam dataset saved successfully!
Corpus passages: 247
Questions: 100


In [36]:
import os

print(os.path.exists("data/corpus_ml.json"))
print(os.path.exists("data/questions_ml.json"))

True
True


In [37]:
import random

random.seed(123)

for i, q in enumerate(random.sample(questions_ml, 5), 1):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    print("Question:", q["question"])
    print("Gold answer:", q["gold_answers"])
    print("Gold passage ID:", q["gold_passage_id"])
    print("Passage:")
    print(corpus_ml[q["gold_passage_id"]]["text"])
    print()

EXAMPLE 1
Question: നവോത്ഥാന നായകന്മാരായ ശ്രീനാരായണ ഗുരുവിന്റെയും അയ്യങ്കാളിയുടെയും കേന്ദ്രം  ഏതായിരുന്നു ?
Gold answer: ['കൊല്ലവും']
Gold passage ID: 44
Passage:
അദ്ദേഹം പുതിയ ചന്തകൾ നിർമ്മിക്കുകയും തമിഴ്‌നാട്ടിലെ മദ്രാസ്, തിരുനൽവേലി എന്നിവിടങ്ങളിൽ നിന്നുള്ള കച്ചവടക്കാരെ കൊല്ലത്ത് വ്യാപാരത്തിനായി ക്ഷണിക്കുകയും ചെയ്തു. ഇതേത്തുടർന്ന് കശുവണ്ടി, കയർ, സുഗന്ധവ്യഞ്ജനങ്ങൾ എന്നിവയുടെ കച്ചവടം കൊല്ലത്ത് തഴച്ചു. ഇക്കാലയളവിലെ കൊല്ലത്തിന്റെ മേന്മകണ്ടാണ് കൊല്ലം കണ്ടവന് ഇല്ലം വേണ്ട എന്ന ചൊല്ല് ഉണ്ടായത്. 1811 റസിഡൻറ് മൺറോയ്ക്കുവേണ്ടി പണിയിപ്പിച്ചതാണ് ആശ്രാമം എന്ന സ്ഥലത്തെ കൊല്ലം റസിഡൻസി. ആതർ എന്ന എൻജിനീയർ ആണ് ഇതിന് നേതൃത്വം കൊടുത്തത്. റസിഡൻറിന്റെ ആസ്ഥാനം, ദിവാൻ കച്ചേരി, അപ്പീൽകോടതി തുടങ്ങിയവയെല്ലാം ആദ്യം കൊല്ലത്തായിരുന്നു. 1803 മുതൽ 1830 വരെ ഇംഗ്ലീഷ് പട്ടാളം തമ്പടിച്ചിരുന്നത് കൊല്ലം കൻറോൺമെൻറിലാണ്. 1809-ൽ ബ്രിട്ടീഷ് ഈസ്റ്റ് ഇന്ത്യാ കമ്പനിയും തിരിവിതാംകൂറും തമ്മിൽ  കൊല്ലം യുദ്ധം നടന്നു. സ്വാതി തിരുനാളിന്റെ കാലത്തോടെയാണ് ദിവാൻ കച്ചേരി തലസ്ഥാനത്തേക്ക് മാറ്റിയത്. തിരുവനന്തപുരത്ത് നായർ ബ്രിഗേഡ് ശക്തി പ്രാപ

In [38]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/corpus_ml.json
	data/questions_ml.json

nothing added to commit but untracked files present (use "git add" to track)


In [39]:
!git add .

In [42]:
!git commit -m "Add Malayalam corpus and questions"

[nehna 62b6b8b] Add Malayalam corpus and questions
 2 files changed, 1792 insertions(+)
 create mode 100644 data/corpus_ml.json
 create mode 100644 data/questions_ml.json


In [43]:
!git push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 334.33 KiB | 2.14 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   56993ba..62b6b8b  nehna -> nehna


In [41]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehranmk638-dev"